# Proyecto de PLN: Normalización y Lematización de Texto
## Caso de Estudio: Don Quijote de la Mancha (Miguel de Cervantes)

Este cuaderno implementa el pipeline de preprocesamiento de lenguaje natural visto en clase:
1. **Tokenización**: Descomposición del flujo de texto en unidades discretas.
2. **Filtrado de Ruido**: Eliminación de palabras vacías (stop words), signos de puntuación y espacios.
3. **Lematización y Normalización (Case Folding)**: Reducción morfológica a lemas canónicos en minúsculas.
4. **Comparativa**: Stemming (NLTK Snowball) vs Lematización (spaCy).
5. **Reducción de Dimensionalidad**: Análisis de la contracción del vocabulario.
6. **Vectorización**: Representación numérica con Bag-of-Words y TF-IDF.
7. **Espacio Vectorial 3D**: Reducción PCA a 3 dimensiones y gráfico comparativo.


In [ ]:
import os
import re
import pandas as pd
import spacy
from nltk.stem import SnowballStemmer

print("Librerías importadas correctamente.")


In [ ]:
# Cargar modelo en español de spaCy
try:
    nlp = spacy.load("es_core_news_sm")
    print("Modelo es_core_news_sm cargado exitosamente.")
except OSError:
    print("Descargando modelo...")
    from spacy.cli import download
    download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")


In [ ]:
# Carga del libro y extracción del Capítulo Primero
with open("don_quijote.txt", "r", encoding="utf-8") as f:
    texto_completo = f.read()

# Extraer el Capítulo Primero para análisis enfocado
patron = r"Capítulo primero\..*?(?=Capítulo II\.|\Z)"
match = re.search(patron, texto_completo, re.DOTALL | re.IGNORECASE)
texto_cap1 = match.group(0).strip() if match else texto_completo[:10000]

print(f"Texto cargado. Longitud: {len(texto_cap1)} caracteres.")
print(f"Fragmento inicial:\n{texto_cap1[:250]}...")


In [ ]:
# 1. TOKENIZACIÓN
doc = nlp(texto_cap1)

print("--- 1. Tokenización ---")
print(f"Total de tokens generados: {len(doc)}")
primeros_tokens = [token.text for token in doc if not token.is_space][:20]
print(f"Muestra de primeros 20 tokens:\n{primeros_tokens}")


In [ ]:
# 2. FILTRADO DE STOP WORDS Y RUIDO
tokens_relevantes = []
tokens_ruido = []

for token in doc:
    if not token.is_stop and not token.is_punct and not token.is_space and token.text.strip():
        tokens_relevantes.append(token.text)
    elif token.is_stop or token.is_punct:
        tokens_ruido.append(token.text)

print("--- 2. Filtrado de Stop Words y Puntuación ---")
print(f"Tokens eliminados (Ruido): {len(tokens_ruido)}")
print(f"Tokens conservados (Contenido útil): {len(tokens_relevantes)}")
print(f"Muestra eliminadas: {tokens_ruido[:10]}")
print(f"Muestra conservadas: {tokens_relevantes[:10]}")


In [ ]:
# 3. LEMATIZACIÓN Y NORMALIZACIÓN
tokens_normalizados = []
cambios_interesantes = []

for token in doc:
    if not token.is_stop and not token.is_punct and not token.is_space and token.text.strip():
        lema = token.lemma_.lower()
        tokens_normalizados.append(lema)
        if token.text.lower() != lema and len(cambios_interesantes) < 10:
            cambios_interesantes.append(f"{token.text} -> {lema}")

print("--- 3. Lematización y Normalización ---")
print(f"Total de tokens normalizados: {len(tokens_normalizados)}")
print("Transformaciones morfológicas (Original -> Lema):")
for c in cambios_interesantes:
    print(f"  * {c}")
print(f"\nMuestra de tokens finales: {tokens_normalizados[:10]}")


In [ ]:
# 4. COMPARATIVA: STEMMING VS LEMATIZACIÓN
stemmer = SnowballStemmer("spanish")
data_comparativa = []

for token in doc:
    if not token.is_punct and not token.is_space and not token.is_stop and token.text.strip():
        raiz_stem = stemmer.stem(token.text)
        lema = token.lemma_.lower()
        data_comparativa.append({
            "Original": token.text,
            "Stemming (NLTK)": raiz_stem,
            "Lematización (spaCy)": lema,
            "Coinciden": raiz_stem == lema
        })

df_comparativa = pd.DataFrame(data_comparativa)
palabras_clave = ["acordarme", "vivía", "antigua", "corredor", "leyendo", "imaginación", "deseaba", "caballeros", "hicieron", "podía"]
filtro = df_comparativa[df_comparativa["Original"].str.lower().isin(palabras_clave)].drop_duplicates(subset=["Original"])

print("--- Comparativa en Palabras Representativas ---")
print(filtro.to_string(index=False))
print("\n--- Primeros 10 Tokens ---")
print(df_comparativa.head(10).to_string(index=False))


In [ ]:
# 5. REDUCCIÓN DE DIMENSIONALIDAD DEL VOCABULARIO
vocabulario_original = len(set([t.text.lower() for t in doc if not t.is_punct and not t.is_space]))
vocabulario_lemas = len(set(tokens_normalizados))
reduccion = ((vocabulario_original - vocabulario_lemas) / vocabulario_original) * 100

resumen = pd.DataFrame({
    "Métrica": ["Tokens Totales", "Tokens Útiles (Sin Ruido)", "Vocabulario Único Original", "Vocabulario Único Lematizado", "Reducción de Dimensionalidad"],
    "Valor": [f"{len(doc):,}", f"{len(tokens_normalizados):,}", f"{vocabulario_original:,}", f"{vocabulario_lemas:,}", f"{reduccion:.2f}%"]
})
print("--- Resumen de Reducción de Dimensionalidad ---")
print(resumen.to_string(index=False))


---
## Checkpoint 4: Representación Vectorial y Semántica

Hasta ahora tenemos lemas limpios, pero siguen siendo texto. Los algoritmos de ML necesitan números.

| Modelo | Idea clave | Ventaja | Desventaja |
|---|---|---|---|
| **Bag-of-Words** | Conteos de términos | Simple, efectivo | Ignora orden; alta sparsity |
| **TF-IDF** | Premia términos distinctivos del documento | Captura relevancia relativa | Igual pierde el orden |


In [ ]:
# 6. CORPUS LEMATIZADO POR ORACIONES
# Cada oración del capítulo se convierte en un documento del corpus
corpus_lematizado = []

for oracion in doc.sents:
    lemas_oracion = [
        token.lemma_.lower()
        for token in oracion
        if not token.is_punct and not token.is_space and not token.is_stop
    ]
    if lemas_oracion:
        corpus_lematizado.append(" ".join(lemas_oracion))

print(f"Total de oraciones procesadas: {len(corpus_lematizado)}")
print(f"\nPrimera oración del corpus:")
print(f"  '{corpus_lematizado[0]}'")


In [ ]:
# 7. BAG-OF-WORDS Y TF-IDF
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Para Bag of Words
bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(corpus_lematizado)

# Para TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(corpus_lematizado)

print(f"Forma de la matriz BoW:    {X_bow.shape}  (oraciones x términos)")
print(f"Forma de la matriz TF-IDF: {X_tfidf.shape}  (oraciones x términos)")
print(f"Vocabulario: {len(bow_vectorizer.vocabulary_)} términos únicos")
print(f"Densidad BoW (no-ceros/total): {X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]):.4f}")

# Top 10 términos TF-IDF de la primera oración
vocab_tfidf = tfidf_vectorizer.get_feature_names_out()
primera = X_tfidf[0].toarray()[0]
top_indices = np.argsort(primera)[::-1][:10]
print(f"\nTop 10 términos TF-IDF (primera oración):")
for i in top_indices:
    if primera[i] > 0:
        print(f"  {vocab_tfidf[i]:<20} peso: {primera[i]:.4f}")


In [ ]:
# 8. VISUALIZACIÓN EN ESPACIO VECTORIAL 3D CON PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA


def graficar_palabras_3d(ax, matriz, vocabulario, titulo, color_puntos):
    # 1. TRANSPONER: Filas = Palabras, Columnas = Contextos
    matriz_palabras = matriz.T
    # 2. PCA a 3 dimensiones
    pca = PCA(n_components=3)
    coords = pca.fit_transform(matriz_palabras.toarray())
    x, y, z = coords[:, 0], coords[:, 1], coords[:, 2]
    # 3. Scatter 3D
    ax.scatter(x, y, z, c=color_puntos, s=80, edgecolors='k', alpha=0.8, depthshade=True)
    for i, palabra in enumerate(vocabulario[:30]):
        ax.text(x[i], y[i], z[i] + 0.01, palabra, fontsize=7)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Comp. Principal 1')
    ax.set_ylabel('Comp. Principal 2')
    ax.set_zlabel('Comp. Principal 3')
    ax.plot([0, 0], [0, 0], [z.min(), z.max()], c='grey', ls='--', lw=0.5, alpha=0.3)
    ax.plot([x.min(), x.max()], [0, 0], [0, 0], c='grey', ls='--', lw=0.5, alpha=0.3)
    ax.plot([0, 0], [y.min(), y.max()], [0, 0], c='grey', ls='--', lw=0.5, alpha=0.3)


fig = plt.figure(figsize=(18, 8))
fig.suptitle('Representación Vectorial del Quijote — Cap. 1', fontsize=14)

ax1 = fig.add_subplot(121, projection='3d')
vocab_bow = bow_vectorizer.get_feature_names_out()
graficar_palabras_3d(ax1, X_bow, vocab_bow, 'Espacio BoW 3D (Conteos)', 'orange')

ax2 = fig.add_subplot(122, projection='3d')
graficar_palabras_3d(ax2, X_tfidf, vocab_tfidf, 'Espacio TF-IDF 3D (Importancia)', 'teal')

plt.tight_layout()
plt.savefig('espacio_vectorial_3d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada como espacio_vectorial_3d.png")


## Conclusiones Técnicas

1. **Eliminación de ruido**: El filtrado de stop words y puntuación redujo significativamente la carga computacional, conservando únicamente las unidades portadoras de significado semántico.
2. **Lematización vs Stemming**: Mientras el stemming recorta sufijos heurísticamente produciendo cadenas no léxicas (ej. *caballer*, *imagin*), la lematización usa análisis morfológico contextual para obtener lemas válidos.
3. **Maldición de la Dimensionalidad**: La normalización y lematización redujo el vocabulario único en más del 30%, mitigando la dispersión de datos.
4. **BoW vs TF-IDF**: BoW asigna el mismo peso a todas las apariciones. TF-IDF penaliza las palabras omnipresentes y premia las *distintivas* de cada documento. En el espacio PCA 3D, TF-IDF muestra mayor separación geométrica reflejando mejor la identidad semántica de cada oración.
5. **Sparsity**: Ambas matrices son extremadamente dispersas ya que ninguna oración contiene todas las palabras del vocabulario. Esto justifica el uso de estructuras `scipy.sparse` en sklearn.
